# Queue windows


## Preliminaries


In [ ]:
# Run me to download latest data from GitHub

from qb_notebook.artifacts import (
    download_and_extract_latest_successful_workflow_artifacts,
)

info = download_and_extract_latest_successful_workflow_artifacts(
    repo="leanprover-community/queueboard-core",
    workflow="upload_backup.yaml",
    out_dir="./data",
    artifact_name="analytics-datasets",
    branch="master",
    search_limit=100,  # change this if you expect there to be > 100 failed runs before the first successful one
)

info

In [ ]:
import polars as pl

from qb_notebook.data_io import load_pr_interval_data, split_queue_windows_by_rule

tables = load_pr_interval_data("data")

df_prs = tables["prs"]
df_events = tables["events"]
df_qw = tables["queue_windows"]

qws = split_queue_windows_by_rule(df_qw, rule_set_ids=(1, 2, 3))
df_qw1 = qws[1]
df_qw2 = qws[2]
df_qw3 = qws[3]

## Queue window setup


In [ ]:
from qb_notebook.intervals import enrich_intervals_with_prs

In [ ]:
from qb_notebook.filters import (
    expr_interval_started_between,
    expr_only_closed,
    expr_pr_has_any_of,
    expr_title_regex,
    filter_rows,
    pr_ids_with_any_labels,
)

In [ ]:
df_qw1_enriched = enrich_intervals_with_prs(df_qw1, df_prs)
print(len(df_qw1_enriched))
df_qw2_enriched = enrich_intervals_with_prs(df_qw2, df_prs)
print(len(df_qw2_enriched))
df_qw3_enriched = enrich_intervals_with_prs(df_qw3, df_prs)
print(len(df_qw3_enriched))

In [ ]:
df_qw1_feat = filter_rows(
    df_qw1_enriched, expr_title_regex(r"(^feat)|(^\[Merged by Bors\] -\s+[fF]eat)")
)
print(len(df_qw1_feat))
df_qw2_feat = filter_rows(
    df_qw2_enriched, expr_title_regex(r"(^feat)|(^\[Merged by Bors\] -\s+[fF]eat)")
)
print(len(df_qw2_feat))
df_qw3_feat = filter_rows(
    df_qw3_enriched, expr_title_regex(r"(^feat)|(^\[Merged by Bors\] -\s+[fF]eat)")
)
print(len(df_qw3_feat))

df_qw1_nonfeat = filter_rows(
    df_qw1_enriched,
    expr_title_regex(r"(^feat)|(^\[Merged by Bors\] -\s+[fF]eat)").not_(),
)
print(len(df_qw1_nonfeat))
df_qw2_nonfeat = filter_rows(
    df_qw2_enriched,
    expr_title_regex(r"(^feat)|(^\[Merged by Bors\] -\s+[fF]eat)").not_(),
)
print(len(df_qw2_nonfeat))
df_qw3_nonfeat = filter_rows(
    df_qw3_enriched,
    expr_title_regex(r"(^feat)|(^\[Merged by Bors\] -\s+[fF]eat)").not_(),
)
print(len(df_qw3_nonfeat))

## PRs on the queue vs time

In [ ]:
from qb_notebook.intervals import effective_queue_prs_per_day

qw_asof = df_qw.select(pl.max("updated_at")).item()

In [ ]:
import altair as alt

daily_queue1 = effective_queue_prs_per_day(df_qw1, asof=qw_asof)
daily_queue1.plot.line(x="day", y="prs_on_queue")

In [ ]:
import altair as alt

daily_queue2 = effective_queue_prs_per_day(df_qw2, asof=qw_asof)
daily_queue2.plot.line(x="day", y="prs_on_queue")

In [ ]:
import altair as alt

daily_queue3 = effective_queue_prs_per_day(df_qw3, asof=qw_asof)
daily_queue3.plot.line(x="day", y="prs_on_queue")

In [ ]:
combined = pl.concat(
    [
        daily_queue1.with_columns(pl.lit("with CI (must pass)").alias("series")),
        daily_queue2.with_columns(pl.lit("without CI").alias("series")),
        daily_queue3.with_columns(pl.lit("with CI (no failures)").alias("series")),
    ]
)

combined.plot.line(
    x="day",
    y="prs_on_queue",
    color="series",
)

In [ ]:
import altair as alt


def daily_queue_counts_with_closed_fallback(df):
    return (
        df.with_columns(
            pl.coalesce([pl.col("to_ts"), pl.col("closed_at"), pl.lit(qw_asof)]).alias(
                "to_ts_effective"
            )
        )
        .select(
            [
                pl.col("pull_request_id"),
                pl.col("from_ts").dt.date().alias("start_day"),
                (pl.col("to_ts_effective") - pl.duration(microseconds=1))
                .dt.date()
                .alias("end_day"),
            ]
        )
        .filter(pl.col("end_day") >= pl.col("start_day"))
        .with_columns(
            pl.date_ranges(
                pl.col("start_day"),
                pl.col("end_day"),
                interval="1d",
                closed="both",
            ).alias("day")
        )
        .explode("day")
        .group_by("day")
        .agg(pl.col("pull_request_id").n_unique().alias("prs_on_queue"))
        .sort("day")
    )


daily_queue1_feat = daily_queue_counts_with_closed_fallback(df_qw1_feat)
daily_queue2_feat = daily_queue_counts_with_closed_fallback(df_qw2_feat)
daily_queue3_feat = daily_queue_counts_with_closed_fallback(df_qw3_feat)
daily_queue1_nonfeat = daily_queue_counts_with_closed_fallback(df_qw1_nonfeat)
daily_queue2_nonfeat = daily_queue_counts_with_closed_fallback(df_qw2_nonfeat)
daily_queue3_nonfeat = daily_queue_counts_with_closed_fallback(df_qw3_nonfeat)

In [ ]:
combined_1_feat_nonfeat = pl.concat(
    [
        daily_queue1_feat.with_columns(pl.lit("with CI, feat").alias("series")),
        daily_queue1_nonfeat.with_columns(pl.lit("with CI, non-feat").alias("series")),
    ]
)

combined_1_feat_nonfeat.plot.line(
    x="day",
    y="prs_on_queue",
    color="series",
)

In [ ]:
combined_2_feat_nonfeat = pl.concat(
    [
        daily_queue2_feat.with_columns(pl.lit("with CI, feat").alias("series")),
        daily_queue2_nonfeat.with_columns(pl.lit("with CI, non-feat").alias("series")),
    ]
)

combined_2_feat_nonfeat.plot.line(
    x="day",
    y="prs_on_queue",
    color="series",
)

## Queue window intervals

In [ ]:
from qb_notebook.intervals import effective_queue_window_durations

In [ ]:
import numpy as np

from qb_notebook.plotting import plot_duration_hist, plot_duration_hists

In [ ]:
df_qw1_dur = effective_queue_window_durations(df_qw1)

df_qw1_closed = df_qw1_dur.filter(pl.col("status") == "closed")
df_qw1_open = df_qw1_dur.filter(pl.col("status") == "open")

df_qw2_dur = effective_queue_window_durations(df_qw2)

df_qw2_closed = df_qw2_dur.filter(pl.col("status") == "closed")
df_qw2_open = df_qw2_dur.filter(pl.col("status") == "open")

In [ ]:
plot_duration_hist(
    df_qw1_closed,
    col="duration_days",
    bins=100,
    logx=True,
    logy=False,
    title="Queue window durations (closed) (with CI)",
)

In [ ]:
plot_duration_hist(
    df_qw2_closed,
    col="duration_days",
    bins=100,
    logx=True,
    logy=False,
    title="Queue window durations (closed) (without CI)",
)

In [ ]:
df_qw1_feat_dur = effective_queue_window_durations(df_qw1_feat)

df_qw1_feat_closed = df_qw1_feat_dur.filter(pl.col("status") == "closed")
df_qw1_feat_open = df_qw1_feat_dur.filter(pl.col("status") == "open")

df_qw2_feat_dur = effective_queue_window_durations(df_qw2_feat)

df_qw2_feat_closed = df_qw2_feat_dur.filter(pl.col("status") == "closed")
df_qw2_feat_open = df_qw2_feat_dur.filter(pl.col("status") == "open")

df_qw1_nonfeat_dur = effective_queue_window_durations(df_qw1_nonfeat)

df_qw1_nonfeat_closed = df_qw1_nonfeat_dur.filter(pl.col("status") == "closed")
df_qw1_nonfeat_open = df_qw1_nonfeat_dur.filter(pl.col("status") == "open")

df_qw2_nonfeat_dur = effective_queue_window_durations(df_qw2_nonfeat)

df_qw2_nonfeat_closed = df_qw2_nonfeat_dur.filter(pl.col("status") == "closed")
df_qw2_nonfeat_open = df_qw2_nonfeat_dur.filter(pl.col("status") == "open")

In [ ]:
plot_duration_hists(
    [
        (df_qw1_feat_closed, "feat"),
        (df_qw1_nonfeat_closed, "non-feat"),
    ],
    col="duration_days",
    logx=True,
    logy=False,
    bins=100,
    title="Queue window durations (closed) (with CI)",
    percentiles=[75, 90],
    percentile_style="vline",
    label_stagger=0.25,
    label_rotate=0,
)

In [ ]:
plot_duration_hists(
    [
        (df_qw2_feat_closed, "feat"),
        (df_qw2_nonfeat_closed, "non-feat"),
    ],
    col="duration_days",
    logx=True,
    logy=False,
    bins=100,
    title="Queue window durations (closed) (without CI)",
    percentiles=[75, 90],
    percentile_style="vline",
    label_stagger=0.25,
    label_rotate=0,
)

## Queue window percentile vs time

In [ ]:
import matplotlib.pyplot as plt

from qb_notebook.intervals import snapshot_queue_age_quantiles

In [ ]:
quantiles = [0.75, 0.90]

pdf = snapshot_queue_age_quantiles(df_qw1, quantiles).to_pandas()

plt.figure()
for q in quantiles:
    col = f"p{int(q * 100)}"
    plt.plot(pdf["date"], pdf[col], label=col)

plt.legend()
plt.xlabel("date (UTC)")
plt.ylabel("age (days) at end-of-day")
plt.title("Queue window age percentiles over time")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
quantiles = [0.75, 0.90]

pdf = snapshot_queue_age_quantiles(df_qw2, quantiles).to_pandas()

plt.figure()
for q in quantiles:
    col = f"p{int(q * 100)}"
    plt.plot(pdf["date"], pdf[col], label=col)

plt.legend()
plt.xlabel("date (UTC)")
plt.ylabel("age (days) at end-of-day")
plt.title("Queue window age percentiles over time")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
quantiles = [0.75, 0.90]

pdf = snapshot_queue_age_quantiles(df_qw3, quantiles, asof=qw_asof).to_pandas()

plt.figure()
for q in quantiles:
    col = f"p{int(q * 100)}"
    plt.plot(pdf["date"], pdf[col], label=col)

plt.legend()
plt.xlabel("date (UTC)")
plt.ylabel("age (days) at end-of-day")
plt.title("Queue window age percentiles over time")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Queue Window Event Attribution

The queue window table now carries attribution columns that record what caused each window boundary:

- `opened_by_event_type` / `closed_by_event_type` — discriminator string (e.g. `CI_PASSED`, `FORBIDDEN_LABEL_REMOVED`, `HEAD_PUSHED`, …)
- `opened_by_timeline_event_id` / `closed_by_timeline_event_id` — FK → `syncer_prtimelineevent`
- `opened_by_check_run_id` / `closed_by_check_run_id` — FK → `syncer_commitcheckrun`
- `opened_by_status_context_id` / `closed_by_status_context_id` — FK → `syncer_commitstatuscontext`
- `opened_at_head_sha` / `closed_at_head_sha` — commit SHA at boundary

At most one FK column is non-null for a given row; `event_type` is the discriminator.
Synthetic boundaries (`INITIAL_STATE`, `RULESET_EFFECTIVE`) have null FKs.

In [ ]:
# Attribution tables loaded by load_pr_interval_data (re-use already-loaded `tables`)
df_check_runs = tables["check_runs"]
df_status_contexts = tables["status_contexts"]

# Confirm FK columns are now Int64 (not Float64)
attr_cols = [
    c for c in df_qw3.columns if "opened_by" in c or "closed_by" in c or "head_sha" in c
]
df_qw3.select(attr_cols).schema

In [ ]:
# Overview: how often does each event type open / close a queue window?
print("opened_by_event_type:")
print(df_qw3["opened_by_event_type"].value_counts(sort=True))
print()
print("closed_by_event_type (null = window still open):")
print(df_qw3["closed_by_event_type"].value_counts(sort=True))

In [ ]:
from qb_notebook.filters import expr_opened_by_event_type, expr_closed_by_event_type

# Convenience: filter df_qw3 by opening / closing event type
df_qw3_ci_opened = filter_rows(df_qw3, expr_opened_by_event_type(["CI_PASSED"]))
df_qw3_label_opened = filter_rows(
    df_qw3, expr_opened_by_event_type(["FORBIDDEN_LABEL_REMOVED"])
)
df_qw3_ci_closed = filter_rows(df_qw3, expr_closed_by_event_type(["CI_FAILED"]))
df_qw3_label_closed = filter_rows(
    df_qw3, expr_closed_by_event_type(["FORBIDDEN_LABEL_ADDED"])
)
df_qw3_head_closed = filter_rows(df_qw3, expr_closed_by_event_type(["HEAD_PUSHED"]))

print("qw3 opened by CI_PASSED:", len(df_qw3_ci_opened))
print("qw3 opened by FORBIDDEN_LABEL_REMOVED:", len(df_qw3_label_opened))
print("qw3 closed by CI_FAILED:", len(df_qw3_ci_closed))
print("qw3 closed by FORBIDDEN_LABEL_ADDED:", len(df_qw3_label_closed))
print("qw3 closed by HEAD_PUSHED:", len(df_qw3_head_closed))

### Joining with timeline events (label / draft / push boundaries)

In [ ]:
# Join queue windows with the timeline event that opened them.
# Only rows where opened_by_timeline_event_id is non-null have a FK.
df_qw_open_tl = df_qw3.filter(pl.col("opened_by_timeline_event_id").is_not_null()).join(
    df_events.select(
        pl.col("id").alias("opened_by_timeline_event_id"),
        pl.col("type").alias("tl_open_type"),
        pl.col("label_name").alias("tl_open_label"),
        pl.col("actor_login").alias("tl_open_actor"),
        pl.col("occurred_at").alias("tl_open_occurred_at"),
    ),
    on="opened_by_timeline_event_id",
    how="left",
)

# Similarly for closing timeline events
df_qw_close_tl = df_qw3.filter(
    pl.col("closed_by_timeline_event_id").is_not_null()
).join(
    df_events.select(
        pl.col("id").alias("closed_by_timeline_event_id"),
        pl.col("type").alias("tl_close_type"),
        pl.col("label_name").alias("tl_close_label"),
        pl.col("actor_login").alias("tl_close_actor"),
        pl.col("occurred_at").alias("tl_close_occurred_at"),
    ),
    on="closed_by_timeline_event_id",
    how="left",
)

print("Windows with opening timeline event:", len(df_qw_open_tl))
print("Windows with closing timeline event:", len(df_qw_close_tl))

In [ ]:
# Which labels most commonly close a window (FORBIDDEN_LABEL_ADDED events)?
df_qw_close_tl.filter(pl.col("closed_by_event_type") == "FORBIDDEN_LABEL_ADDED")[
    "tl_close_label"
].value_counts(sort=True).head(20)

In [ ]:
# Which labels most commonly open a window (FORBIDDEN_LABEL_REMOVED events)?
df_qw_open_tl.filter(pl.col("opened_by_event_type") == "FORBIDDEN_LABEL_REMOVED")[
    "tl_open_label"
].value_counts(sort=True).head(20)

### Joining with CI tables (check runs / status contexts)

In [ ]:
# Join CI_PASSED / CI_FAILED windows with the triggering check run.
# Note: CI events may point to check_run OR status_context, not both.
df_qw_open_cr = df_qw3.filter(pl.col("opened_by_check_run_id").is_not_null()).join(
    df_check_runs.select(
        pl.col("id").alias("opened_by_check_run_id"),
        pl.col("name").alias("cr_open_name"),
        pl.col("conclusion").alias("cr_open_conclusion"),
    ),
    on="opened_by_check_run_id",
    how="left",
)

df_qw_close_cr = df_qw3.filter(pl.col("closed_by_check_run_id").is_not_null()).join(
    df_check_runs.select(
        pl.col("id").alias("closed_by_check_run_id"),
        pl.col("name").alias("cr_close_name"),
        pl.col("conclusion").alias("cr_close_conclusion"),
    ),
    on="closed_by_check_run_id",
    how="left",
)

print("Windows opened by a check run:", len(df_qw_open_cr))
print("Windows closed by a check run:", len(df_qw_close_cr))

In [ ]:
# Which check run names most commonly open a window (CI_PASSED)?
df_qw_open_cr["cr_open_name"].value_counts(sort=True).head(20)

In [ ]:
# Which check run names / conclusions most commonly close a window (CI_FAILED)?
df_qw_close_cr.group_by(["cr_close_name", "cr_close_conclusion"]).len().sort(
    "len", descending=True
).head(20)

### Queue window duration by event type

In [ ]:
import matplotlib.pyplot as plt

from qb_notebook.intervals import effective_queue_window_durations

df_qw3_dur_attr = (
    effective_queue_window_durations(df_qw3)
    .filter(pl.col("status") == "closed")
    .filter(pl.col("opened_by_event_type").is_not_null())
)

# Median duration (days) grouped by opened_by_event_type
median_by_open_type = (
    df_qw3_dur_attr.group_by("opened_by_event_type")
    .agg(
        pl.col("duration_days").median().alias("median_days"),
        pl.col("duration_days").mean().alias("mean_days"),
        pl.len().alias("count"),
    )
    .sort("median_days", descending=True)
)
median_by_open_type

In [ ]:
# Box-plot-style: median duration by opened_by_event_type (qw3, closed windows)
pdf = median_by_open_type.to_pandas().sort_values("median_days", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(pdf["opened_by_event_type"], pdf["median_days"], color="steelblue")
for bar, cnt in zip(bars, pdf["count"]):
    ax.text(
        bar.get_width() + 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"n={cnt:,}",
        va="center",
        fontsize=8,
    )
ax.set_xlabel("Median duration (days)")
ax.set_title("Median closed queue-window duration by opened_by_event_type (rule set 3)")
fig.tight_layout()
plt.show()

In [ ]:
# Median duration by closed_by_event_type (excluding still-open windows)
median_by_close_type = (
    df_qw3_dur_attr.filter(pl.col("closed_by_event_type").is_not_null())
    .group_by("closed_by_event_type")
    .agg(
        pl.col("duration_days").median().alias("median_days"),
        pl.len().alias("count"),
    )
    .sort("median_days", descending=True)
)

pdf2 = median_by_close_type.to_pandas().sort_values("median_days", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(pdf2["closed_by_event_type"], pdf2["median_days"], color="coral")
for bar, cnt in zip(bars, pdf2["count"]):
    ax.text(
        bar.get_width() + 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"n={cnt:,}",
        va="center",
        fontsize=8,
    )
ax.set_xlabel("Median duration (days)")
ax.set_title("Median closed queue-window duration by closed_by_event_type (rule set 3)")
fig.tight_layout()
plt.show()